# 🥒 Zucchini Color Scoring — Machine Learning Pipeline

**Task:** Ordinal ranking of zucchini images into 8 color-intensity scores (1 = lightest green → 8 = darkest green)  
**Dataset:** 213 images, farmer-labeled, 8 skewed classes  
**Approach:** Test 8 ML models with rigorous anti-overfitting controls

---

## 📋 Full Pipeline

```
Raw TIF images (Score=01..08 folders)
  │
  ├─ Step 0 : Imports & Configuration
  ├─ Step 1 : Load TIF → uint8 RGB           [load_tif_uint8()]
  ├─ Step 2 : Inscribed-square crop           [get_inscribed_square_crop()]
  ├─ Step 3 : Statistical LAB filter          [stat_filter()]  K=1.5σ — removes specular highlights & noise
  ├─ Step 4 : 27-dim feature extraction       [extract_lab_features()]  from filtered foreground pixels only
  ├─ Step 5 : Feature visualisation           boxplots, heatmap, scatter matrix
  ├─ Step 6 : Leakage-free 70/20/10 split        stratified train/test/val, BEFORE augmentation
  ├─ Step 7 : Colour-safe augmentation        TRAIN only — 15 variants × original = ×16 dataset
  ├─ Step 8 : Feature engineering             variance filter → correlation filter → SelectKBest → StandardScaler
  ├─ Step 9 : Model training + Optuna tuning  8 models, 5-fold stratified CV each
  └─ Step 10: Evaluation & comparison         MAE, MSE, RMSE, Spearman ρ, Acc@1, Acc@2, Confusion Matrix
```

> ⚠️ **Data leakage rules enforced throughout:**
> - Augmentation applied **after** the train/test split — test images are never augmented
> - Feature engineering (scaler, selector) fitted **only** on train folds, then applied to val/test
> - Stratified 5-Fold CV used for all model selection (43 test images is too small for a fixed val set)


## 0 · Imports & Configuration

In [ ]:
pip install optuna

In [ ]:
!pip install numpy pandas scipy opencv-python tifffile scikit-learn matplotlib seaborn optuna mord xgboost tqdm

import os, glob, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import tifffile as tiff
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

from scipy.stats import spearmanr, kendalltau, skew, kurtosis
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, mean_squared_error,
                             mean_absolute_error, r2_score)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.pipeline import Pipeline

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ── Configuration ────────────────────────────────────────────────────────────
DATA_ROOT       = r'F:\Aymen\server\statistical_filtering'
IMG_SIZE        = (256, 256)
K_STAT          = 1.5      # σ half-width for statistical LAB filter
FG_THRESH       = 10       # pixel value threshold to detect foreground
TEST_SPLIT      = 0.20
VAL_SPLIT       = 0.10
N_FOLDS         = 5
N_OPTUNA_TRIALS = 100
OUT_DIR         = 'ml_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

SCORE_FOLDERS = [f'Score={str(i).zfill(2)}' for i in range(1, 9)]
IDX_TO_SCORE  = np.array([1,2,3,4,5,6,7,8], dtype=np.float32)
NUM_CLASSES   = 8

def savefig(name, dpi=130):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  💾 {path}')

print('Configuration ready ✔')
print(f'  DATA_ROOT : {DATA_ROOT}')
print(f'  IMG_SIZE  : {IMG_SIZE}')
print(f'  K_STAT    : {K_STAT}')
print(f'  N_FOLDS   : {N_FOLDS}')
print(f'  Optuna trials per model: {N_OPTUNA_TRIALS}')

## 1 · Load TIF Images

16-bit TIFF images are loaded and normalised to uint8 RGB.  
The score label is parsed directly from the folder name (`Score=01` → score 1).  
Both RGB and LAB representations are stored: RGB for geometric augmentation, LAB for photometric augmentation and feature extraction.


In [ ]:
def load_tif_uint8(path: str) -> np.ndarray:
    """Load uint16 TIFF → normalise to uint8 RGB."""
    img = tiff.imread(path)
    if img.ndim == 4:
        img = img[0]
    if img.ndim == 3 and img.shape[0] in (1, 3, 4):
        img = np.transpose(img, (1, 2, 0))
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    elif img.shape[-1] == 1:
        img = np.concatenate([img]*3, axis=-1)
    elif img.shape[-1] == 4:
        img = img[..., :3]
    if img.dtype != np.uint8:
        lo, hi = img.min(), img.max()
        img = ((img.astype(np.float32) - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)
    return img


def rgb_to_lab(img_rgb: np.ndarray) -> np.ndarray:
    """uint8 RGB → uint8 LAB (OpenCV convention)."""
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)


def collect_images(root: str) -> tuple:
    """Load all TIF images from Score=01..08 sub-folders."""
    rgb_list, lab_list, score_list, path_list = [], [], [], []
    for folder in SCORE_FOLDERS:
        score_int = int(folder.split('=')[1])
        patterns = [os.path.join(root, '**', folder, ext)
                    for ext in ['*.tif','*.TIF','*.tiff','*.TIFF']]
        paths = []
        for pat in patterns:
            paths += glob.glob(pat, recursive=True)
        paths = list(set(paths))
        for p in paths:
            try:
                rgb = load_tif_uint8(p)
                rgb = cv2.resize(rgb, (IMG_SIZE[1], IMG_SIZE[0]), interpolation=cv2.INTER_AREA)
                lab = rgb_to_lab(rgb)
                rgb_list.append(rgb); lab_list.append(lab)
                score_list.append(score_int); path_list.append(p)
            except Exception as e:
                print(f'  ⚠ skip {p}: {e}')
    X_rgb = np.array(rgb_list, dtype=np.uint8)
    X_lab = np.array(lab_list, dtype=np.uint8)
    y     = np.array(score_list, dtype=np.int32)
    df    = pd.DataFrame({'path': path_list, 'score': score_list})
    print(f'Loaded {len(df)} images')
    print(df.groupby('score')['path'].count().rename('count').to_string())
    return X_rgb, X_lab, y, df


X_rgb, X_lab, y_raw, df_all = collect_images(DATA_ROOT)


In [ ]:
# ── Class distribution plot ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
counts = np.bincount(y_raw, minlength=9)[1:]
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, NUM_CLASSES))
bars = ax.bar([f'Score {i+1}' for i in range(NUM_CLASSES)], counts, color=colors, edgecolor='white')
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width()/2, c + 0.3, str(c), ha='center', fontsize=10, fontweight='bold')
ax.set_title('Dataset Class Distribution (213 images, farmer-labeled)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of images'); ax.set_xlabel('Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); savefig('00_class_distribution.png'); plt.show()


## 2 · Inscribed-Square Crop

Each image is cropped to the largest square inscribed in the largest inscribed circle of the foreground region.  
This removes background noise and image borders while keeping only the central zucchini area.  

**Why inscribed-square crop?**  
- Eliminates non-zucchini pixels (borders, shadows, background)
- Produces a consistent square input regardless of original image shape
- Pixels outside the inscribed circle are set to white (neutral, not black, to avoid biasing LAB stats)


In [ ]:
def get_inscribed_square_crop(img: np.ndarray) -> tuple:
    """
    Inscribed-square crop:
    1. Detect foreground pixels (any channel > FG_THRESH).
    2. Distance transform → largest inscribed circle centre.
    3. Crop the largest square inscribed in that circle; corners → white.
    Returns: (cropped_img, info_dict)
    """
    H, W = img.shape[:2]
    fg_mask = np.any(img > FG_THRESH, axis=-1).astype(np.uint8)
    dist = cv2.distanceTransform(fg_mask, cv2.DIST_L2, 5)
    _, r_f, _, (cx, cy) = cv2.minMaxLoc(dist)
    r = int(r_f); cx, cy = int(cx), int(cy)
    side = int(r * 2**0.5); side = side - (side % 2)
    sx1 = max(0, cx - side//2); sy1 = max(0, cy - side//2)
    sx2 = min(W, sx1 + side);   sy2 = min(H, sy1 + side)
    side = min(sx2-sx1, sy2-sy1); sx2, sy2 = sx1+side, sy1+side
    patch = img[sy1:sy2, sx1:sx2].copy()
    gy, gx = np.mgrid[sy1:sy2, sx1:sx2]
    outside = ((gx - cx)**2 + (gy - cy)**2) > r**2
    patch[outside] = 255
    return patch, {'cx':cx,'cy':cy,'r':r,'side':side,'sx1':sx1,'sy1':sy1,'sx2':sx2,'sy2':sy2}


# Apply to all images
X_rgb_cropped = []
crop_infos    = []
for img in tqdm(X_rgb, desc='Cropping'):
    c, info = get_inscribed_square_crop(img)
    X_rgb_cropped.append(cv2.resize(c, (IMG_SIZE[1], IMG_SIZE[0]), interpolation=cv2.INTER_AREA))
    crop_infos.append(info)
X_rgb_cropped = np.array(X_rgb_cropped, dtype=np.uint8)
X_lab_cropped = np.array([rgb_to_lab(r) for r in X_rgb_cropped], dtype=np.uint8)
print(f'Cropped: {X_rgb_cropped.shape}')


In [ ]:
# ── Crop visualisation: one sample per score ─────────────────────────────────
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(20, 5))
for i in range(NUM_CLASSES):
    idx = np.where(y_raw == i+1)[0][0]
    axes[0, i].imshow(X_rgb[idx])
    axes[0, i].set_title(f'Score {i+1}\nOriginal', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(X_rgb_cropped[idx])
    axes[1, i].set_title('Cropped', fontsize=8)
    axes[1, i].axis('off')
plt.suptitle('Inscribed-Square Crop — One Sample per Score', fontsize=12, fontweight='bold')
plt.tight_layout(); savefig('01_crop_visualisation.png'); plt.show()


## 3 · Statistical LAB Filter

For each cropped image, foreground pixels are kept only if their L\*, a\*, b\* values fall within `[mean ± K·σ]` (K=1.5).  
Rejected pixels are set to white. This removes:
- **Specular highlights** (overexposed bright spots)
- **Shadow pixels** (too dark, not representative of surface colour)
- **Noise pixels** at the distribution extremes

**Design choice:** K=1.5 (tighter than K=2.0) to aggressively remove non-representative pixels,  
keeping only the clean, well-lit green surface area.


In [ ]:
def stat_filter(img_rgb: np.ndarray, k: float = K_STAT) -> dict:
    """
    Statistical LAB interval filter on foreground pixels.
    Keeps pixels where each LAB channel is within [mean ± k·σ].
    Rejected pixels → white.
    """
    H, W = img_rgb.shape[:2]
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[...,0] * 100.0 / 255.0
    a_img = lab[...,1] - 128.0
    b_img = lab[...,2] - 128.0
    coords = np.argwhere(fg_mask)
    if len(coords) == 0:
        return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0}
    r, c = coords[:,0], coords[:,1]
    keep_px = np.ones(len(coords), dtype=bool)
    for ch_img in [L_img, a_img, b_img]:
        vals = ch_img[r, c]
        med, std = vals.mean(), vals.std()
        keep_px &= (vals >= med - k*std) & (vals <= med + k*std)
    kept_mask = np.zeros((H, W), bool)
    kept_mask[r[keep_px], c[keep_px]] = True
    out = np.full_like(img_rgb, 255)
    out[kept_mask] = img_rgb[kept_mask]
    return {'out': out, 'kept_mask': kept_mask,
            'kept_pct': float(keep_px.sum() / len(coords) * 100)}


# Apply filter to all cropped images
filter_results = [stat_filter(img) for img in tqdm(X_rgb_cropped, desc='Filtering')]
X_rgb_filtered = np.array([r['out']        for r in filter_results], dtype=np.uint8)
X_masks        = np.array([r['kept_mask']  for r in filter_results], dtype=bool)
kept_pcts      = np.array([r['kept_pct']   for r in filter_results], dtype=np.float32)

print(f'Kept pixel %: mean={kept_pcts.mean():.1f}%  std={kept_pcts.std():.1f}%  '
      f'min={kept_pcts.min():.1f}%  max={kept_pcts.max():.1f}%')


In [ ]:
# ── Filter visualisation: crop vs filtered for 2 samples per score ───────────
fig, axes = plt.subplots(3, NUM_CLASSES, figsize=(20, 7))
row_labels = ['Original crop', f'Filtered (K={K_STAT}σ)', 'Kept mask']
for i in range(NUM_CLASSES):
    idx = np.where(y_raw == i+1)[0][0]
    axes[0,i].imshow(X_rgb_cropped[idx])
    axes[0,i].set_title(f'Score {i+1}', fontsize=8, fontweight='bold')
    axes[1,i].imshow(X_rgb_filtered[idx])
    axes[1,i].set_title(f'{kept_pcts[idx]:.0f}% kept', fontsize=8)
    axes[2,i].imshow(X_masks[idx], cmap='Greens')
    for ax in axes[:,i]: ax.axis('off')
for ax, lbl in zip(axes[:,0], row_labels):
    ax.set_ylabel(lbl, fontsize=9, fontweight='bold')
plt.suptitle(f'Statistical LAB Filter (K=±{K_STAT}σ) — One Sample per Score',
             fontsize=12, fontweight='bold')
plt.tight_layout(); savefig('02_filter_visualisation.png'); plt.show()


## 4 · 27-dim Feature Extraction

A 27-dimensional feature vector is computed **from filtered foreground pixels only** (clean green surface).

Layout per LAB channel (×3 channels = 27 total):
```
[mean, std, P10, P25, P50, P75, P90, skew, kurtosis]  →  9 dims × 3 = 27
```

**Why these features?**
- Mean and std capture the central tendency and spread of each channel
- Percentiles (P10–P90) capture the full distribution shape without histogram binning
- Skewness and kurtosis describe distribution asymmetry and tail behaviour
- Computed in **physical-scale LAB** (L∈[0,100], a,b∈[−128,128]) for interpretability
- The `a*` channel directly encodes the green↔red axis — the primary colour signal


In [ ]:
def extract_lab_features(img_rgb: np.ndarray, fg_mask: np.ndarray) -> np.ndarray:
    """
    27-dim feature vector from filtered foreground pixels.
    Per channel (L*, a*, b*): mean, std, P10, P25, P50, P75, P90, skew, kurtosis
    9 x 3 = 27 features. Physical-scale LAB.
    """
    lab   = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[...,0] * 100.0 / 255.0
    a_img = lab[...,1] - 128.0
    b_img = lab[...,2] - 128.0
    feats = []
    for ch in [L_img, a_img, b_img]:
        vals = ch[fg_mask]
        if len(vals) == 0:
            feats.extend([0.0] * 9)
            continue
        feats.append(float(vals.mean()))
        feats.append(float(vals.std()))
        feats.extend(np.percentile(vals, [10, 25, 50, 75, 90]).tolist())
        feats.append(float(skew(vals)))
        feats.append(float(kurtosis(vals)))
    return np.array(feats, dtype=np.float32)


feat_names = []
for ch in ['L*', 'a*', 'b*']:
    feat_names += [f'{ch}_mean', f'{ch}_std',
                   f'{ch}_P10', f'{ch}_P25', f'{ch}_P50',
                   f'{ch}_P75', f'{ch}_P90',
                   f'{ch}_skew', f'{ch}_kurt']

# Extract for all filtered images
X_feats = np.array([
    extract_lab_features(rgb, mask)
    for rgb, mask in tqdm(zip(X_rgb_filtered, X_masks), total=len(X_masks), desc='Features')
], dtype=np.float32)

print(f'Feature matrix: {X_feats.shape}  (N x 27)')
print(f'Feature names : {feat_names}')


## 5 · Feature Visualisation

Before training, we inspect the 27 features to understand which ones carry the most discriminative signal  
across the 8 score classes. This also helps sanity-check the pipeline (e.g. a*_mean should decrease  
monotonically from Score 1 to Score 8 as the green becomes darker/more saturated).


In [ ]:
# ── Per-class boxplots for 6 key features ────────────────────────────────────
key_idx  = [0, 9, 18, 1, 10, 19]
key_name = ['L* mean','a* mean','b* mean','L* std','a* std','b* std']
colors   = plt.cm.RdYlGn(np.linspace(0.1, 0.9, NUM_CLASSES))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, fi, fname in zip(axes.ravel(), key_idx, key_name):
    per_class = [X_feats[y_raw == s+1, fi] for s in range(NUM_CLASSES)]
    bp = ax.boxplot(per_class, patch_artist=True, notch=False)
    for patch, col in zip(bp['boxes'], colors):
        patch.set_facecolor(col)
    ax.set_xticklabels([f'S{i+1}' for i in range(NUM_CLASSES)], fontsize=9)
    ax.set_title(fname, fontsize=11, fontweight='bold')
    ax.set_xlabel('Score class'); ax.grid(axis='y', alpha=0.3)
plt.suptitle('LAB Feature Distributions per Score Class', fontsize=14, fontweight='bold')
plt.tight_layout(); savefig('03_feature_distributions.png'); plt.show()


In [ ]:
# -- Feature correlation heatmap: before & after |r| > 0.95 reduction ------
corr_full = pd.DataFrame(X_feats, columns=feat_names).corr()

# Apply correlation filter
corr_np = np.abs(np.corrcoef(X_feats.T))
n_feat = X_feats.shape[1]
keep_mask = np.ones(n_feat, dtype=bool)
for i in range(n_feat):
    if not keep_mask[i]: continue
    for j in range(i+1, n_feat):
        if keep_mask[j] and corr_np[i, j] > 0.95:
            keep_mask[j] = False

removed_feats = [feat_names[i] for i in range(n_feat) if not keep_mask[i]]
kept_feats    = [feat_names[i] for i in range(n_feat) if keep_mask[i]]

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# Before
tri1 = np.triu(np.ones_like(corr_full, dtype=bool))
sns.heatmap(corr_full, mask=tri1, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            ax=axes[0], linewidths=0.3, annot=False,
            xticklabels=feat_names, yticklabels=feat_names)
axes[0].set_title(f'Before reduction ({len(feat_names)} features)', fontsize=11, fontweight='bold')
axes[0].tick_params(labelsize=6)

# After
X_reduced = X_feats[:, keep_mask]
corr_red  = pd.DataFrame(X_reduced, columns=kept_feats).corr()
tri2 = np.triu(np.ones_like(corr_red, dtype=bool))
sns.heatmap(corr_red, mask=tri2, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            ax=axes[1], linewidths=0.3, annot=True, fmt='.2f', annot_kws={'size': 6},
            xticklabels=kept_feats, yticklabels=kept_feats)
axes[1].set_title(f'After |r| > 0.95 reduction ({len(kept_feats)} features)',
                  fontsize=11, fontweight='bold')
axes[1].tick_params(labelsize=7)

plt.suptitle('Feature Correlation Matrix - Before & After Reduction',
             fontsize=13, fontweight='bold')
plt.tight_layout(); savefig('04_correlation_heatmap.png'); plt.show()

print(f'Features before: {len(feat_names)}')
print(f'Features after : {len(kept_feats)}')
print(f'Removed ({len(removed_feats)}): {removed_feats}')


In [ ]:
# ── Scatter: a* mean vs L* mean, coloured by score ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
scatter_colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, NUM_CLASSES))
cmap_scores    = plt.cm.RdYlGn
sc = axes[0].scatter(X_feats[:,0], X_feats[:,9], c=y_raw, cmap=cmap_scores,
                     vmin=1, vmax=8, alpha=0.8, edgecolors='white', linewidths=0.3, s=60)
axes[0].set_xlabel('L* mean'); axes[0].set_ylabel('a* mean')
axes[0].set_title('L* mean vs a* mean', fontweight='bold'); plt.colorbar(sc, ax=axes[0], label='Score')

sc2 = axes[1].scatter(X_feats[:,9], X_feats[:,18], c=y_raw, cmap=cmap_scores,
                      vmin=1, vmax=8, alpha=0.8, edgecolors='white', linewidths=0.3, s=60)
axes[1].set_xlabel('a* mean'); axes[1].set_ylabel('b* mean')
axes[1].set_title('a* mean vs b* mean', fontweight='bold'); plt.colorbar(sc2, ax=axes[1], label='Score')
plt.suptitle('Feature Scatter — Coloured by Score', fontsize=12, fontweight='bold')
plt.tight_layout(); savefig('05_feature_scatter.png'); plt.show()


## 6 · Leakage-Free Train / Test Split

> ⚠️ **Critical leakage rule:** The split is performed on **original images** before any augmentation.
> Test images are **never augmented**, **never used** to fit any transformer or selector.

Stratified **80/20** split ensures proportional representation across all score classes:
- **Train (80%):** augmented ×16, used for CV training and hyperparameter tuning
- **Test (20%):** held-out final evaluation — never seen during training or tuning

**5-Fold Stratified CV** on the training set is used for all model selection decisions.
CV folds are split by **original image identity** — all augmented variants of one image stay together
in the same fold. Validation folds use **only original (non-augmented)** images to prevent leakage.


In [ ]:
idx_all = np.arange(len(X_feats))

# 80% train / 20% test — stratified
idx_tr, idx_te = train_test_split(
    idx_all, test_size=0.20, stratify=y_raw, random_state=SEED)

X_feats_tr_orig = X_feats[idx_tr];  y_tr_orig = y_raw[idx_tr]
X_rgb_tr_orig   = X_rgb_filtered[idx_tr]
X_lab_tr_orig   = np.array([rgb_to_lab(r) for r in X_rgb_tr_orig])

X_feats_te  = X_feats[idx_te];   y_te  = y_raw[idx_te]

print(f'Train (original): {len(idx_tr)} images  ({len(idx_tr)/len(idx_all)*100:.0f}%)')
print(f'Test            : {len(idx_te)} images  ({len(idx_te)/len(idx_all)*100:.0f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lbl, color, y_split in [
    (axes[0], 'Train',  'steelblue', y_tr_orig),
    (axes[1], 'Test',   'seagreen',  y_te)]:
    cnt = np.bincount(y_split, minlength=9)[1:]
    bars = ax.bar([f'S{i+1}' for i in range(NUM_CLASSES)], cnt, color=color, edgecolor='white')
    for b, c in zip(bars, cnt):
        ax.text(b.get_x()+b.get_width()/2, c+0.2, str(c), ha='center', fontsize=9)
    ax.set_title(f'{lbl} ({len(y_split)} images)', fontweight='bold')
    ax.set_ylim(0, cnt.max()+5); ax.grid(axis='y', alpha=0.3)
plt.suptitle('Stratified 80/20 Split (Before Augmentation)', fontsize=12, fontweight='bold')
plt.tight_layout(); savefig('06_split_distribution.png'); plt.show()


## 7 · Colour-Safe Augmentation (Train Only)

Augmentation is applied **only to training images** to prevent data leakage.  

**Allowed augmentations** (colour-identity preserving):

| Type | Variants | Applied on | Rationale |
|---|---|---|---|
| Flip | H / V / HV | RGB | Does not change colour |
| Rotation | +15° / −15° / +30° | RGB | Preserves LAB values |
| Crop/Zoom | +10% / +20% / +30% | RGB | Spatial only |
| Blur | σ=0.5 / 1.0 / 1.5 | LAB | Smooths texture, keeps colour |
| Noise | σ=3 / 6 / 9 | LAB | Adds robustness |

**Forbidden augmentations:** `HueSaturationValue`, `ColorJitter` with hue/sat, `RandomGamma`  
— these directly perturb the colour intensity being measured, creating false labels.

After each augmentation: stat_filter is re-applied and features re-extracted from filtered pixels.  
This multiplies the training set by **×16** (1 original + 15 variants).


In [ ]:
# ── Augmentation functions ───────────────────────────────────────────────────
def aug_flip(rgb, v):
    return cv2.flip(rgb, [1, 0, -1][v])

def aug_rotate(rgb, v):
    angles = [15, -15, 30]
    h, w   = rgb.shape[:2]
    M      = cv2.getRotationMatrix2D((w/2, h/2), angles[v], 1.0)
    return cv2.warpAffine(rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def aug_zoom(rgb, v):
    pcts = [0.10, 0.20, 0.30]
    f    = pcts[v]; h, w = rgb.shape[:2]
    dh, dw = int(h*f/2), int(w*f/2)
    return cv2.resize(rgb[dh:h-dh, dw:w-dw], (w, h), interpolation=cv2.INTER_LINEAR)

def aug_blur(lab, v):
    return cv2.GaussianBlur(lab, (3, 3), [0.5, 1.0, 1.5][v])

def aug_noise(lab, v):
    rng = np.random.default_rng(SEED + v)
    n   = rng.normal(0, [3,6,9][v], lab.shape).astype(np.float32)
    return np.clip(lab.astype(np.float32) + n, 0, 255).astype(np.uint8)

AUG_CATALOGUE = [
    ('Flip',     aug_flip,   'rgb', 3),
    ('Rotation', aug_rotate, 'rgb', 3),
    ('Zoom',     aug_zoom,   'rgb', 3),
    ('Blur',     aug_blur,   'lab', 3),
    ('Noise',    aug_noise,  'lab', 3),
]
print(f'Augmentation variants per image: {sum(n for *_,n in AUG_CATALOGUE)}')
print(f'Training set multiplier: ×{1 + sum(n for *_,n in AUG_CATALOGUE)}')


In [ ]:
# ── Build augmented training set with origin tracking ────────────────
aug_feats_list, aug_y_list = [], []
aug_orig_idx_list = []   # maps each sample to its original train image index
aug_is_original_list = [] # True only for the original (non-augmented) sample

for i, (rgb, lab, lbl) in enumerate(tqdm(
        zip(X_rgb_tr_orig, X_lab_tr_orig, y_tr_orig),
        total=len(y_tr_orig), desc='Augmenting')):
    # Original
    res = stat_filter(rgb)
    aug_feats_list.append(extract_lab_features(res['out'], res['kept_mask']))
    aug_y_list.append(lbl)
    aug_orig_idx_list.append(i)
    aug_is_original_list.append(True)
    # Augmented variants
    for aug_name, aug_fn, domain, n_variants in AUG_CATALOGUE:
        for v in range(n_variants):
            if domain == 'rgb':
                aug_rgb = aug_fn(rgb, v)
                aug_lab = rgb_to_lab(aug_rgb)
            else:
                aug_lab = aug_fn(lab, v)
                aug_rgb = cv2.cvtColor(aug_lab, cv2.COLOR_LAB2RGB)
            res = stat_filter(aug_rgb)
            aug_feats_list.append(extract_lab_features(res['out'], res['kept_mask']))
            aug_y_list.append(lbl)
            aug_orig_idx_list.append(i)
            aug_is_original_list.append(False)

X_feats_tr_aug  = np.array(aug_feats_list, dtype=np.float32)
y_tr_aug        = np.array(aug_y_list,     dtype=np.int32)
aug_orig_idx    = np.array(aug_orig_idx_list, dtype=np.int32)
aug_is_original = np.array(aug_is_original_list, dtype=bool)

print(f'Augmented training set: {X_feats_tr_aug.shape}')
print(f'Original samples: {aug_is_original.sum()}, Augmented: {(~aug_is_original).sum()}')
print('Per-class counts:')
for s in range(1, 9):
    print(f'  Score {s}: {(y_tr_aug == s).sum()}')


In [ ]:
# ── Augmentation grid (one row per score, one col per variant) ───────────────
aug_labels = (['Original'] +
              [f'Flip-{i+1}' for i in range(3)] +
              [f'Rot-{i+1}'  for i in range(3)] +
              [f'Zoom-{i+1}' for i in range(3)] +
              [f'Blur-{i+1}' for i in range(3)] +
              [f'Noise-{i+1}'for i in range(3)])
border_cols = (['#333']+['#2980b9']*3+['#27ae60']*3+['#8e44ad']*3+['#16a085']*3+['#c0392b']*3)
n_cols = len(aug_labels)
fig, axes = plt.subplots(NUM_CLASSES, n_cols, figsize=(n_cols*1.4, NUM_CLASSES*1.7))
for cls_i in range(NUM_CLASSES):
    idxs = np.where(y_tr_orig == cls_i+1)[0]
    if len(idxs) == 0: continue
    rgb0 = X_rgb_tr_orig[idxs[0]]; lab0 = X_lab_tr_orig[idxs[0]]
    display = [rgb0]
    for _, aug_fn, domain, nv in AUG_CATALOGUE:
        for v in range(nv):
            if domain == 'rgb':
                display.append(aug_fn(rgb0, v))
            else:
                display.append(cv2.cvtColor(aug_fn(lab0, v), cv2.COLOR_LAB2RGB))
    for col, (ax, img, bc) in enumerate(zip(axes[cls_i], display, border_cols)):
        ax.imshow(img)
        if cls_i == 0: ax.set_title(aug_labels[col], fontsize=5.5, color=bc, fontweight='bold')
        if col == 0:   ax.set_ylabel(f'S{cls_i+1}', fontsize=8, fontweight='bold')
        ax.axis('off')
        for sp in ax.spines.values(): sp.set_edgecolor(bc); sp.set_linewidth(1.5)
plt.suptitle('Colour-Safe Augmentation Grid per Score Class', fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout(); savefig('07_augmentation_grid.png'); plt.show()


## 8 · Feature Engineering

Applied in this order, **fitted only on training data** to prevent leakage:

1. **Variance filter** — remove features with near-zero variance (σ² < 0.01)
2. **Correlation filter** — remove one of each pair with Pearson |r| > 0.95
3. **SelectKBest** (f_classif) — keep top K most discriminative features; K tuned by Optuna
4. **StandardScaler** — zero-mean, unit-variance normalisation

The full transformer chain is fit on each CV training fold independently,  
then applied to the val fold — **never fit on val or test data**.


In [ ]:
def correlation_filter(X: np.ndarray, threshold: float = 0.95) -> np.ndarray:
    # Return boolean mask of features to KEEP.
    # For each pair with |r| > threshold, drop the higher-index feature.
    # Fitted on X (training data only).
    corr   = np.corrcoef(X.T)
    n_feat = X.shape[1]
    keep   = np.ones(n_feat, dtype=bool)
    for i in range(n_feat):
        if not keep[i]: continue
        for j in range(i+1, n_feat):
            if keep[j] and abs(corr[i, j]) > threshold:
                keep[j] = False
    return keep


# Replace NaN in features (from skew/kurtosis on tiny pixel sets)
X_feats_tr_aug = np.nan_to_num(X_feats_tr_aug, nan=0.0)
X_feats        = np.nan_to_num(X_feats, nan=0.0)
X_feats_te     = np.nan_to_num(X_feats_te, nan=0.0)
print(f'NaN cleanup done. Remaining NaN in train: {np.isnan(X_feats_tr_aug).sum()}')

# Preview feature selection on full training set (display only)
vt_preview   = VarianceThreshold(threshold=0.01).fit(X_feats_tr_aug)
X_vt         = vt_preview.transform(X_feats_tr_aug)
corr_mask    = correlation_filter(X_vt, threshold=0.95)
X_corr       = X_vt[:, corr_mask]
kb_preview   = SelectKBest(f_classif, k=min(20, X_corr.shape[1])).fit(X_corr, y_tr_aug)
X_kb         = kb_preview.transform(X_corr)

print(f'Original features      : {len(feat_names)}')
print(f'After variance filter  : {X_vt.shape[1]}')
print(f'After correlation filter: {X_corr.shape[1]}')
print(f'After SelectKBest (k=20): {X_kb.shape[1]}')

# F-score bar chart
feat_names_vt   = np.array(feat_names)[vt_preview.get_support()]
feat_names_corr = feat_names_vt[corr_mask]
f_scores        = kb_preview.scores_
n_top = min(20, len(f_scores))
top_idx         = np.argsort(f_scores)[::-1][:n_top]
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(top_idx)), f_scores[top_idx], color='steelblue', edgecolor='white')
ax.set_xticks(range(len(top_idx)))
ax.set_xticklabels([feat_names_corr[i] for i in top_idx], rotation=45, ha='right', fontsize=8)
ax.set_title(f'Top {len(top_idx)} Feature F-Scores (SelectKBest, f_classif) — fitted on train set', fontweight='bold')
ax.set_ylabel('F-score'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); savefig('08_feature_importance.png'); plt.show()


## 9 · Model Training & Hyperparameter Tuning

### Strategy

- **5-Fold Stratified CV** on the augmented training set for all model selection
- **Optuna** (50 trials per model) tunes hyperparameters — primary CV objective is Spearman ρ
- Final model trained on the full augmented training set with best hyperparameters
- Evaluated on the clean held-out test set (never seen during training or tuning)

### Evaluation metrics

| Metric | What it measures |
|---|---|
| MAE | Mean absolute error in score units |
| MSE | Mean squared error |
| RMSE | Root mean squared error |
| Spearman ρ | Rank correlation (primary — robust to monotone nonlinearity) |
| Kendall τ | Another rank correlation |
| Acc@1 | % predictions within 1 score step of true (ordinal tolerance) |
| Acc@2 | % predictions within 2 score steps of true |
| Confusion matrix | Full class-level breakdown |


In [ ]:
# ── Shared evaluation helper ─────────────────────────────────────────────────
def evaluate(y_true: np.ndarray, y_pred_cont: np.ndarray) -> dict:
    """Compute all metrics given continuous predictions and integer true scores."""
    y_pred_r = np.clip(np.round(y_pred_cont), 1, 8).astype(int)
    y_true   = np.array(y_true, dtype=int)
    mae  = mean_absolute_error(y_true, y_pred_cont)
    mse  = mean_squared_error(y_true, y_pred_cont)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred_cont)
    rho, _  = spearmanr(y_true, y_pred_cont)
    tau, _  = kendalltau(y_true, y_pred_cont)
    acc0 = float(np.mean(y_pred_r == y_true))
    acc1 = float(np.mean(np.abs(y_pred_r - y_true) <= 1))
    cm   = confusion_matrix(y_true, y_pred_r, labels=list(range(1, 9)))
    return {'MAE':mae,'MSE':mse,'RMSE':rmse,'R2':r2,
            'Spearman_rho':rho,'Kendall_tau':tau,'Acc@0':acc0,'Acc@1':acc1,'CM':cm}


# ── Feature pipeline builder (called inside each CV fold) ────────────────────
def build_feature_pipeline(X_train, y_train, k_best: int):
    """Fit the full feature pipeline on training fold ONLY."""
    X_train = np.nan_to_num(X_train, nan=0.0)
    vt   = VarianceThreshold(threshold=0.01).fit(X_train)
    Xv   = vt.transform(X_train)
    cmask = correlation_filter(Xv, 0.95)
    Xc   = Xv[:, cmask]
    k    = min(k_best, Xc.shape[1])
    kb   = SelectKBest(f_classif, k=k).fit(Xc, y_train)
    Xk   = kb.transform(Xc)
    sc   = StandardScaler().fit(Xk)
    Xf   = sc.transform(Xk)
    return Xf, (vt, cmask, kb, sc)


def apply_feature_pipeline(X, transformers):
    """Apply a pre-fitted pipeline to new data (val/test)."""
    vt, cmask, kb, sc = transformers
    X = np.nan_to_num(X, nan=0.0)
    Xv = vt.transform(X)
    Xc = Xv[:, cmask]
    Xk = kb.transform(Xc)
    return sc.transform(Xk)


# ── Augmentation-aware cross-validation runner ───────────────────────────────
def cv_objective(trial, model_fn, X, y, k_best=20):
    """
    5-fold stratified CV with augmentation-aware splitting.
    Returns COMPOSITE score: 0.50·ρ + 0.25·R² + 0.25·Acc@1
    to prevent overfitting to rank-only or point-error-only.
    """
    n_orig = int(aug_orig_idx.max()) + 1
    y_orig_labels = y[aug_is_original]
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scores = []
    for tr_orig, va_orig in skf.split(np.arange(n_orig), y_orig_labels):
        tr_mask = np.isin(aug_orig_idx, tr_orig)
        va_mask = np.isin(aug_orig_idx, va_orig) & aug_is_original
        X_tr, y_tr = X[tr_mask], y[tr_mask]
        X_va, y_va = X[va_mask], y[va_mask]
        Xf_tr, pipes = build_feature_pipeline(X_tr, y_tr, k_best)
        Xf_va        = apply_feature_pipeline(X_va, pipes)
        model = model_fn(trial)
        model.fit(Xf_tr, y_tr)
        preds = model.predict(Xf_va)
        preds = np.clip(preds, 1, 8)
        rho, _ = spearmanr(y_va, preds)
        r2     = r2_score(y_va, preds)
        y_pred_r = np.clip(np.round(preds), 1, 8).astype(int)
        acc1   = float(np.mean(np.abs(y_pred_r - y_va) <= 1))
        # Composite: balances ranking, point accuracy, and exact prediction
        composite = 0.50 * max(0, rho) + 0.25 * max(0, r2) + 0.25 * acc1
        scores.append(composite)
    return float(np.mean(scores)) if scores else 0.0


results = {}
print('Helpers ready ✔  (augmentation-aware CV, composite objective)')
print(f'  CV objective: 0.50·ρ + 0.25·R² + 0.25·Acc@1')


### 9.1 · Ordinal Logistic Regression (mord)

**What it is:** All-Thresholds ordinal logistic regression — learns 7 thresholds on a single latent axis,  
forcing the model to respect the 1→8 ordering structurally.  

**Why it fits this task:** Unlike standard classification, it never treats score 3 and score 7 as equally "wrong" for score 5.  
It is also very sample-efficient — important with only 213 images.


In [ ]:
from mord import LogisticAT

def tune_ordinal_lr():
    def objective(trial):
        k = trial.suggest_int('k_best', 8, 50)
        alpha = trial.suggest_float('alpha', 1e-5, 50.0, log=True)
        def model_fn(t):
            return LogisticAT(alpha=alpha, max_iter=2000)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning Ordinal Logistic Regression...')
best_olr = tune_ordinal_lr()
print('Best params:', best_olr)

k_olr = best_olr['k_best']
Xf_tr, pipes_olr = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_olr)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_olr)
model_olr = LogisticAT(alpha=best_olr['alpha'], max_iter=2000)
model_olr.fit(Xf_tr, y_tr_aug)
preds_olr = model_olr.predict(Xf_te).astype(float)
results['Ordinal LR'] = {'metrics': evaluate(y_te, preds_olr),
                          'best_params': best_olr, 'preds': preds_olr}
m = results['Ordinal LR']['metrics']
print(f"  MAE={m['MAE']:.3f}  R²={m['R2']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@0={m['Acc@0']:.2%}  Acc@1={m['Acc@1']:.2%}")


In [ ]:
%pip install mord --quiet

from sklearn.linear_model import HuberRegressor

def tune_huber():
    def objective(trial):
        k       = trial.suggest_int('k_best', 8, 50)
        epsilon = trial.suggest_float('epsilon', 1.05, 4.0)
        alpha   = trial.suggest_float('alpha', 1e-6, 5.0, log=True)
        def model_fn(t):
            return HuberRegressor(epsilon=epsilon, alpha=alpha, max_iter=500)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning Huber Regression...')
best_hub = tune_huber()
print('Best params:', best_hub)

k_hub = best_hub['k_best']
Xf_tr, pipes_hub = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_hub)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_hub)
model_hub = HuberRegressor(epsilon=best_hub['epsilon'], alpha=best_hub['alpha'], max_iter=500)
model_hub.fit(Xf_tr, y_tr_aug)
preds_hub = np.clip(model_hub.predict(Xf_te), 1, 8)
results['Huber Regression'] = {'metrics': evaluate(y_te, preds_hub),
                                'best_params': best_hub, 'preds': preds_hub}
m = results['Huber Regression']['metrics']
print(f"  MAE={m['MAE']:.3f}  R²={m['R2']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@0={m['Acc@0']:.2%}  Acc@1={m['Acc@1']:.2%}")


In [ ]:
from sklearn.linear_model import HuberRegressor

def tune_huber():
    def objective(trial):
        k       = trial.suggest_int('k_best', 10, 40)
        epsilon = trial.suggest_float('epsilon', 1.1, 3.0)
        alpha   = trial.suggest_float('alpha', 1e-5, 1.0, log=True)
        def model_fn(t):
            return HuberRegressor(epsilon=epsilon, alpha=alpha, max_iter=300)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning Huber Regression...')
best_hub = tune_huber()
print('Best params:', best_hub)

k_hub = best_hub['k_best']
Xf_tr, pipes_hub = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_hub)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_hub)
model_hub = HuberRegressor(epsilon=best_hub['epsilon'], alpha=best_hub['alpha'], max_iter=300)
model_hub.fit(Xf_tr, y_tr_aug)
preds_hub = np.clip(model_hub.predict(Xf_te), 1, 8)
results['Huber Regression'] = {'metrics': evaluate(y_te, preds_hub),
                                'best_params': best_hub, 'preds': preds_hub}
m = results['Huber Regression']['metrics']
print(f"  MAE={m['MAE']:.3f}  RMSE={m['RMSE']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@1={m['Acc@1']:.2%}")


from sklearn.ensemble import RandomForestRegressor

def tune_rf():
    def objective(trial):
        k    = trial.suggest_int('k_best', 8, 50)
        n_est = trial.suggest_int('n_estimators', 100, 500)
        depth = trial.suggest_int('max_depth', 3, 15)
        msl   = trial.suggest_int('min_samples_leaf', 2, 15)
        mss   = trial.suggest_int('min_samples_split', 4, 20)
        mf    = trial.suggest_categorical('max_features', ['sqrt','log2', 0.5, 0.7])
        ccp   = trial.suggest_float('ccp_alpha', 0.0, 0.05)
        def model_fn(t):
            return RandomForestRegressor(n_estimators=n_est, max_depth=depth,
                                         min_samples_leaf=msl, min_samples_split=mss,
                                         max_features=mf, ccp_alpha=ccp,
                                         random_state=SEED, n_jobs=-1)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning Random Forest...')
best_rf = tune_rf()
print('Best params:', best_rf)

k_rf = best_rf['k_best']
Xf_tr, pipes_rf = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_rf)
Xf_te           = apply_feature_pipeline(X_feats_te, pipes_rf)
model_rf = RandomForestRegressor(n_estimators=best_rf['n_estimators'],
                                  max_depth=best_rf['max_depth'],
                                  min_samples_leaf=best_rf['min_samples_leaf'],
                                  min_samples_split=best_rf['min_samples_split'],
                                  max_features=best_rf['max_features'],
                                  ccp_alpha=best_rf['ccp_alpha'],
                                  random_state=SEED, n_jobs=-1)
model_rf.fit(Xf_tr, y_tr_aug)
preds_rf = np.clip(model_rf.predict(Xf_te), 1, 8)
results['Random Forest'] = {'metrics': evaluate(y_te, preds_rf),
                             'best_params': best_rf, 'preds': preds_rf}
m = results['Random Forest']['metrics']
print(f"  MAE={m['MAE']:.3f}  R²={m['R2']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@0={m['Acc@0']:.2%}  Acc@1={m['Acc@1']:.2%}")


In [ ]:
from sklearn.ensemble import RandomForestRegressor

def tune_rf():
    def objective(trial):
        k    = trial.suggest_int('k_best', 10, 40)
        n_est = trial.suggest_int('n_estimators', 50, 300)
        depth = trial.suggest_int('max_depth', 3, 20)
        msl   = trial.suggest_int('min_samples_leaf', 1, 10)
        mf    = trial.suggest_categorical('max_features', ['sqrt','log2'])
        def model_fn(t):
            return RandomForestRegressor(n_estimators=n_est, max_depth=depth,
                                         min_samples_leaf=msl, max_features=mf,
                                         random_state=SEED, n_jobs=-1)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning Random Forest...')
best_rf = tune_rf()
print('Best params:', best_rf)

k_rf = best_rf['k_best']
Xf_tr, pipes_rf = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_rf)
Xf_te           = apply_feature_pipeline(X_feats_te, pipes_rf)
model_rf = RandomForestRegressor(n_estimators=best_rf['n_estimators'],
                                  max_depth=best_rf['max_depth'],
                                  min_samples_leaf=best_rf['min_samples_leaf'],
                                  max_features=best_rf['max_features'],
                                  random_state=SEED, n_jobs=-1)
model_rf.fit(Xf_tr, y_tr_aug)
preds_rf = np.clip(model_rf.predict(Xf_te), 1, 8)
results['Random Forest'] = {'metrics': evaluate(y_te, preds_rf),
                             'best_params': best_rf, 'preds': preds_rf}
m = results['Random Forest']['metrics']
print(f"  MAE={m['MAE']:.3f}  RMSE={m['RMSE']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@1={m['Acc@1']:.2%}")


import xgboost as xgb

def tune_xgb():
    def objective(trial):
        k    = trial.suggest_int('k_best', 8, 50)
        n_est = trial.suggest_int('n_estimators', 50, 500)
        depth = trial.suggest_int('max_depth', 2, 8)
        lr    = trial.suggest_float('learning_rate', 0.005, 0.3, log=True)
        sub   = trial.suggest_float('subsample', 0.5, 1.0)
        col   = trial.suggest_float('colsample_bytree', 0.4, 1.0)
        lmbd  = trial.suggest_float('reg_lambda', 0.1, 50.0, log=True)
        alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True)
        mcw   = trial.suggest_int('min_child_weight', 1, 10)
        gamma = trial.suggest_float('gamma', 0.0, 5.0)
        def model_fn(t):
            return xgb.XGBRegressor(n_estimators=n_est, max_depth=depth,
                                    learning_rate=lr, subsample=sub,
                                    colsample_bytree=col, reg_lambda=lmbd,
                                    reg_alpha=alpha, min_child_weight=mcw,
                                    gamma=gamma,
                                    random_state=SEED, n_jobs=-1,
                                    eval_metric='mae', verbosity=0)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning XGBoost...')
best_xgb = tune_xgb()
print('Best params:', best_xgb)

k_xgb = best_xgb['k_best']
Xf_tr, pipes_xgb = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_xgb)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_xgb)
model_xgb = xgb.XGBRegressor(
    n_estimators=best_xgb['n_estimators'], max_depth=best_xgb['max_depth'],
    learning_rate=best_xgb['learning_rate'], subsample=best_xgb['subsample'],
    colsample_bytree=best_xgb['colsample_bytree'], reg_lambda=best_xgb['reg_lambda'],
    reg_alpha=best_xgb['reg_alpha'], min_child_weight=best_xgb['min_child_weight'],
    gamma=best_xgb['gamma'],
    random_state=SEED, n_jobs=-1, eval_metric='mae', verbosity=0)
model_xgb.fit(Xf_tr, y_tr_aug)
preds_xgb = np.clip(model_xgb.predict(Xf_te), 1, 8)
results['XGBoost'] = {'metrics': evaluate(y_te, preds_xgb),
                      'best_params': best_xgb, 'preds': preds_xgb}
m = results['XGBoost']['metrics']
print(f"  MAE={m['MAE']:.3f}  R²={m['R2']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@0={m['Acc@0']:.2%}  Acc@1={m['Acc@1']:.2%}")


In [ ]:
import xgboost as xgb

def tune_xgb():
    def objective(trial):
        k    = trial.suggest_int('k_best', 10, 40)
        n_est = trial.suggest_int('n_estimators', 50, 300)
        depth = trial.suggest_int('max_depth', 2, 8)
        lr    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        sub   = trial.suggest_float('subsample', 0.6, 1.0)
        col   = trial.suggest_float('colsample_bytree', 0.5, 1.0)
        lmbd  = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)
        def model_fn(t):
            return xgb.XGBRegressor(n_estimators=n_est, max_depth=depth,
                                    learning_rate=lr, subsample=sub,
                                    colsample_bytree=col, reg_lambda=lmbd,
                                    random_state=SEED, n_jobs=-1,
                                    eval_metric='mae', verbosity=0)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning XGBoost...')
best_xgb = tune_xgb()
print('Best params:', best_xgb)

k_xgb = best_xgb['k_best']
Xf_tr, pipes_xgb = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_xgb)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_xgb)
model_xgb = xgb.XGBRegressor(
    n_estimators=best_xgb['n_estimators'], max_depth=best_xgb['max_depth'],
    learning_rate=best_xgb['learning_rate'], subsample=best_xgb['subsample'],
    colsample_bytree=best_xgb['colsample_bytree'], reg_lambda=best_xgb['reg_lambda'],
    random_state=SEED, n_jobs=-1, eval_metric='mae', verbosity=0)
model_xgb.fit(Xf_tr, y_tr_aug)
preds_xgb = np.clip(model_xgb.predict(Xf_te), 1, 8)
results['XGBoost'] = {'metrics': evaluate(y_te, preds_xgb),
                      'best_params': best_xgb, 'preds': preds_xgb}
m = results['XGBoost']['metrics']
print(f"  MAE={m['MAE']:.3f}  RMSE={m['RMSE']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@1={m['Acc@1']:.2%}")


from sklearn.neighbors import KNeighborsRegressor

def tune_knn():
    def objective(trial):
        k      = trial.suggest_int('k_best', 8, 50)
        n_neigh= trial.suggest_int('n_neighbors', 2, 20)
        weights= trial.suggest_categorical('weights', ['uniform','distance'])
        metric = trial.suggest_categorical('metric', ['euclidean','manhattan','minkowski'])
        p      = trial.suggest_int('p', 1, 4) if metric == 'minkowski' else 2
        def model_fn(t):
            return KNeighborsRegressor(n_neighbors=n_neigh, weights=weights,
                                       metric=metric, p=p)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning KNN...')
best_knn = tune_knn()
print('Best params:', best_knn)

k_knn = best_knn['k_best']
Xf_tr, pipes_knn = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_knn)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_knn)
model_knn = KNeighborsRegressor(n_neighbors=best_knn['n_neighbors'],
                                 weights=best_knn['weights'], metric=best_knn['metric'])
model_knn.fit(Xf_tr, y_tr_aug)
preds_knn = np.clip(model_knn.predict(Xf_te), 1, 8)
results['KNN'] = {'metrics': evaluate(y_te, preds_knn),
                  'best_params': best_knn, 'preds': preds_knn}
m = results['KNN']['metrics']
print(f"  MAE={m['MAE']:.3f}  R²={m['R2']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@0={m['Acc@0']:.2%}  Acc@1={m['Acc@1']:.2%}")


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

def tune_knn():
    def objective(trial):
        k      = trial.suggest_int('k_best', 10, 40)
        n_neigh= trial.suggest_int('n_neighbors', 3, 15)
        weights= trial.suggest_categorical('weights', ['uniform','distance'])
        metric = trial.suggest_categorical('metric', ['euclidean','manhattan'])
        def model_fn(t):
            return KNeighborsRegressor(n_neighbors=n_neigh, weights=weights, metric=metric)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning KNN...')
best_knn = tune_knn()
print('Best params:', best_knn)

k_knn = best_knn['k_best']
Xf_tr, pipes_knn = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_knn)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_knn)
model_knn = KNeighborsRegressor(n_neighbors=best_knn['n_neighbors'],
                                 weights=best_knn['weights'], metric=best_knn['metric'])
model_knn.fit(Xf_tr, y_tr_aug)
preds_knn = np.clip(model_knn.predict(Xf_te), 1, 8)
results['KNN'] = {'metrics': evaluate(y_te, preds_knn),
                  'best_params': best_knn, 'preds': preds_knn}
m = results['KNN']['metrics']
print(f"  MAE={m['MAE']:.3f}  RMSE={m['RMSE']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@1={m['Acc@1']:.2%}")


from sklearn.naive_bayes import GaussianNB

def tune_gnb():
    def objective(trial):
        k    = trial.suggest_int('k_best', 8, 50)
        vs   = trial.suggest_float('var_smoothing', 1e-12, 1e-5, log=True)
        def model_fn(t):
            return GaussianNB(var_smoothing=vs)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning Gaussian Naive Bayes...')
best_gnb = tune_gnb()
print('Best params:', best_gnb)

k_gnb = best_gnb['k_best']
Xf_tr, pipes_gnb = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_gnb)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_gnb)
model_gnb = GaussianNB(var_smoothing=best_gnb['var_smoothing'])
model_gnb.fit(Xf_tr, y_tr_aug)
preds_gnb = model_gnb.predict(Xf_te).astype(float)
results['Gaussian NB'] = {'metrics': evaluate(y_te, preds_gnb),
                           'best_params': best_gnb, 'preds': preds_gnb}
m = results['Gaussian NB']['metrics']
print(f"  MAE={m['MAE']:.3f}  R²={m['R2']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@0={m['Acc@0']:.2%}  Acc@1={m['Acc@1']:.2%}")


In [ ]:
from sklearn.naive_bayes import GaussianNB

def tune_gnb():
    def objective(trial):
        k    = trial.suggest_int('k_best', 10, 40)
        vs   = trial.suggest_float('var_smoothing', 1e-11, 1e-7, log=True)
        def model_fn(t):
            return GaussianNB(var_smoothing=vs)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning Gaussian Naive Bayes...')
best_gnb = tune_gnb()
print('Best params:', best_gnb)

k_gnb = best_gnb['k_best']
Xf_tr, pipes_gnb = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_gnb)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_gnb)
model_gnb = GaussianNB(var_smoothing=best_gnb['var_smoothing'])
model_gnb.fit(Xf_tr, y_tr_aug)
preds_gnb = model_gnb.predict(Xf_te).astype(float)
results['Gaussian NB'] = {'metrics': evaluate(y_te, preds_gnb),
                           'best_params': best_gnb, 'preds': preds_gnb}
m = results['Gaussian NB']['metrics']
print(f"  MAE={m['MAE']:.3f}  RMSE={m['RMSE']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@1={m['Acc@1']:.2%}")


### 9.7 · Pairwise Ranking SVM (RankSVM)

**What it is:** Transforms the ranking problem into binary classification on image pairs.  
For each pair (i, j) where `score_i > score_j`, we create a feature vector `X_i − X_j` with label +1.  
A LinearSVC learns to classify which image in a pair has the higher score.  
The final ranking score is computed by a **tournament**: each test image is scored by  
how many other test images it "wins" against.

**Why it fits:** Pairwise training multiplies data points significantly and directly  
encodes the ordinal cost — getting a score-1 vs score-8 pair wrong contributes more training signal  
than a score-4 vs score-5 pair (controlled by the `|score_i − score_j| >= 1` filter).


In [ ]:
from sklearn.svm import LinearSVC

def build_pairs(X: np.ndarray, y: np.ndarray, min_gap: int = 1, max_pairs: int = 50000):
    # Build pairwise training data.
    X_pairs, y_pairs = [], []
    n = len(y)
    for i in range(n):
        for j in range(i+1, n):
            if (y[i] - y[j]) >= min_gap:
                X_pairs.append(X[i] - X[j]); y_pairs.append(1)
                X_pairs.append(X[j] - X[i]); y_pairs.append(-1)
            elif (y[j] - y[i]) >= min_gap:
                X_pairs.append(X[j] - X[i]); y_pairs.append(1)
                X_pairs.append(X[i] - X[j]); y_pairs.append(-1)
    X_p = np.array(X_pairs, dtype=np.float32)
    y_p = np.array(y_pairs, dtype=np.int32)
    if max_pairs and len(X_p) > max_pairs:
        rng = np.random.default_rng(SEED)
        sel = rng.choice(len(X_p), max_pairs, replace=False)
        X_p, y_p = X_p[sel], y_p[sel]
    return X_p, y_p


def rank_predict(model, X_test: np.ndarray) -> np.ndarray:
    # Vectorized tournament scoring using decision_function.
    n = len(X_test)
    wins = np.zeros(n, dtype=float)
    for i in range(n):
        diffs = X_test[i] - X_test  # (n, d)
        mask = np.ones(n, dtype=bool); mask[i] = False
        scores = model.decision_function(diffs[mask])
        wins[i] = (scores > 0).sum()
    mn, mx = wins.min(), wins.max()
    if mx > mn:
        wins = 1.0 + 7.0 * (wins - mn) / (mx - mn)
    else:
        wins = np.full(n, 4.5)
    return wins


def tune_ranksvm():
    def objective(trial):
        k = trial.suggest_int('k_best', 10, 40)
        C = trial.suggest_float('C', 0.001, 10.0, log=True)
        # Augmentation-aware CV: split by original image identity
        n_orig = int(aug_orig_idx.max()) + 1
        y_orig_labels = y_tr_aug[aug_is_original]
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        rhos = []
        for tr_orig, va_orig in skf.split(np.arange(n_orig), y_orig_labels):
            tr_mask = np.isin(aug_orig_idx, tr_orig)
            va_mask = np.isin(aug_orig_idx, va_orig) & aug_is_original
            X_tr, y_tr = X_feats_tr_aug[tr_mask], y_tr_aug[tr_mask]
            X_va, y_va = X_feats_tr_aug[va_mask], y_tr_aug[va_mask]
            Xf_tr, pipes = build_feature_pipeline(X_tr, y_tr, k)
            Xf_va        = apply_feature_pipeline(X_va, pipes)
            Xp, yp       = build_pairs(Xf_tr, y_tr, min_gap=1)
            if len(np.unique(yp)) < 2: continue
            clf = LinearSVC(C=C, max_iter=2000, random_state=SEED)
            clf.fit(Xp, yp)
            preds = rank_predict(clf, Xf_va)
            rho, _ = spearmanr(y_va, preds)
            rhos.append(rho)
        return float(np.mean(rhos)) if rhos else 0.0

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning Pairwise Ranking SVM...')
best_rank = tune_ranksvm()
print('Best params:', best_rank)

k_rank = best_rank['k_best']
Xf_tr, pipes_rank = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_rank)
Xf_te             = apply_feature_pipeline(X_feats_te, pipes_rank)
Xp_tr, yp_tr      = build_pairs(Xf_tr, y_tr_aug, min_gap=1)
model_rank = LinearSVC(C=best_rank['C'], max_iter=2000, random_state=SEED)
model_rank.fit(Xp_tr, yp_tr)
preds_rank = rank_predict(model_rank, Xf_te)
results['Pairwise RankSVM'] = {'metrics': evaluate(y_te, preds_rank),
                                'best_params': best_rank, 'preds': preds_rank}
m = results['Pairwise RankSVM']['metrics']
print(f"  MAE={m['MAE']:.3f}  RMSE={m['RMSE']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@1={m['Acc@1']:.2%}")


import time
from sklearn.svm import SVR

def tune_svr():
    def objective(trial):
        k       = trial.suggest_int('k_best', 8, 50)
        C       = trial.suggest_float('C', 0.01, 200.0, log=True)
        gamma   = trial.suggest_float('gamma', 1e-5, 2.0, log=True)
        epsilon = trial.suggest_float('epsilon', 0.005, 1.5)
        def model_fn(t):
            return SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon,
                       max_iter=10000, cache_size=500)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning SVR (RBF kernel)...')
t_svr_start = time.time()
best_svr = tune_svr()
t_svr_tune = time.time() - t_svr_start
print(f'Tuning done in {t_svr_tune:.1f}s ({t_svr_tune/60:.1f} min)')
print('Best params:', best_svr)

k_svr = best_svr['k_best']
Xf_tr, pipes_svr = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_svr)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_svr)
model_svr = SVR(kernel='rbf', C=best_svr['C'], gamma=best_svr['gamma'],
                epsilon=best_svr['epsilon'], max_iter=10000, cache_size=500)
model_svr.fit(Xf_tr, y_tr_aug)
preds_svr = np.clip(model_svr.predict(Xf_te), 1, 8)
results['SVR'] = {'metrics': evaluate(y_te, preds_svr),
                  'best_params': best_svr, 'preds': preds_svr}
m = results['SVR']['metrics']
print(f"  MAE={m['MAE']:.3f}  R²={m['R2']:.3f}  ρ={m['Spearman_rho']:.4f}  Acc@0={m['Acc@0']:.2%}  Acc@1={m['Acc@1']:.2%}")


In [ ]:
import time
from sklearn.svm import SVR

def tune_svr():
    def objective(trial):
        k       = trial.suggest_int('k_best', 10, 40)
        C       = trial.suggest_float('C', 0.01, 100.0, log=True)
        gamma   = trial.suggest_float('gamma', 1e-4, 1.0, log=True)
        epsilon = trial.suggest_float('epsilon', 0.01, 1.0)
        def model_fn(t):
            return SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon,
                       max_iter=5000, cache_size=500)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug, k_best=k)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

print('Tuning SVR (RBF kernel, max_iter=5000)...')
t_svr_start = time.time()
best_svr = tune_svr()
t_svr_tune = time.time() - t_svr_start
print(f'Tuning done in {t_svr_tune:.1f}s ({t_svr_tune/60:.1f} min)')
print('Best params:', best_svr)

# Fit final model on full augmented training set
k_svr = best_svr['k_best']
Xf_tr, pipes_svr = build_feature_pipeline(X_feats_tr_aug, y_tr_aug, k_svr)
Xf_te            = apply_feature_pipeline(X_feats_te, pipes_svr)
model_svr = SVR(kernel='rbf', C=best_svr['C'], gamma=best_svr['gamma'],
                epsilon=best_svr['epsilon'], max_iter=5000, cache_size=500)
model_svr.fit(Xf_tr, y_tr_aug)
preds_svr = np.clip(model_svr.predict(Xf_te), 1, 8)
results['SVR'] = {'metrics': evaluate(y_te, preds_svr),
                  'best_params': best_svr, 'preds': preds_svr}
m = results['SVR']['metrics']
print(f"  MAE={m['MAE']:.3f}  RMSE={m['RMSE']:.3f}  rho={m['Spearman_rho']:.4f}  Acc@1={m['Acc@1']:.2%}")

## 10 · Evaluation & Comparison

All 8 models are now evaluated on the **clean held-out test set** (original, never-augmented images).  
Results are compared across all metrics to identify the best approach for this task.


In [ ]:
# ── 10.1 Metrics summary table ───────────────────────────────────────────────
rows = []
for name, res in results.items():
    m = res['metrics']
    rows.append({
        'Model': name,
        'MAE':   round(m['MAE'],   3),
        'RMSE':  round(m['RMSE'],  3),
        'R²':    round(m['R2'],    3),
        'Spearman ρ': round(m['Spearman_rho'], 4),
        'Kendall τ':  round(m['Kendall_tau'],  4),
        'Acc@0': f"{m['Acc@0']:.1%}",
        'Acc@1': f"{m['Acc@1']:.1%}",
    })
df_results = pd.DataFrame(rows).sort_values('Spearman ρ', ascending=False).reset_index(drop=True)
print('\n📊 Model Comparison (sorted by Spearman ρ)')
print(df_results.to_string(index=False))


In [ ]:
# ── 10.2 Confusion matrices (one per model) ──────────────────────────────────
n_models = len(results)
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
score_labels_cm = [str(i) for i in range(1, 9)]
cmap_cm = sns.color_palette('Blues', as_cmap=True)

for ax, (name, res) in zip(axes.ravel(), results.items()):
    cm = res['metrics']['CM']
    rho = res['metrics']['Spearman_rho']
    mae = res['metrics']['MAE']
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap_cm, ax=ax,
                xticklabels=score_labels_cm, yticklabels=score_labels_cm,
                linewidths=0.5, cbar=False)
    ax.set_title(f'{name}\nρ={rho:.3f}  MAE={mae:.2f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted Score', fontsize=8)
    ax.set_ylabel('True Score', fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle('Confusion Matrices — All Models (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout(); savefig('09_confusion_matrices.png'); plt.show()


In [ ]:
# ── 10.3 MAE / RMSE / R² / Accuracy bar chart comparison ─────────────────────
model_names = list(results.keys())
maes  = [results[n]['metrics']['MAE']  for n in model_names]
rmses = [results[n]['metrics']['RMSE'] for n in model_names]
r2s   = [results[n]['metrics']['R2']   for n in model_names]
acc0s = [results[n]['metrics']['Acc@0'] for n in model_names]
acc1s = [results[n]['metrics']['Acc@1'] for n in model_names]

x = np.arange(len(model_names)); w = 0.28
fig, axes = plt.subplots(1, 3, figsize=(22, 5))

# Errors
axes[0].bar(x - w/2, maes,  w, label='MAE',  color='#e74c3c', edgecolor='white')
axes[0].bar(x + w/2, rmses, w, label='RMSE', color='#e67e22', edgecolor='white')
for i, (m, r) in enumerate(zip(maes, rmses)):
    axes[0].text(i - w/2, m + 0.01, f'{m:.2f}', ha='center', fontsize=7)
    axes[0].text(i + w/2, r + 0.01, f'{r:.2f}', ha='center', fontsize=7)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names, rotation=35, ha='right', fontsize=8)
axes[0].set_ylabel('Error (score units)'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
axes[0].set_title('MAE & RMSE (lower is better)', fontweight='bold')

# R²
axes[1].bar(x, r2s, 0.5, color='#3498db', edgecolor='white')
for i, v in enumerate(r2s):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names, rotation=35, ha='right', fontsize=8)
axes[1].set_ylabel('R²'); axes[1].grid(axis='y', alpha=0.3)
axes[1].set_title('R² (higher is better)', fontweight='bold')
axes[1].set_ylim(min(0, min(r2s) - 0.1), 1.05)

# Accuracy
axes[2].bar(x - w/2, acc0s, w, label='Acc@0 (exact)', color='#2c3e50', edgecolor='white')
axes[2].bar(x + w/2, acc1s, w, label='Acc@1 (within-1)', color='#27ae60', edgecolor='white')
for i, (a0, a1) in enumerate(zip(acc0s, acc1s)):
    axes[2].text(i - w/2, a0 + 0.005, f'{a0:.0%}', ha='center', fontsize=7)
    axes[2].text(i + w/2, a1 + 0.005, f'{a1:.0%}', ha='center', fontsize=7)
axes[2].set_xticks(x); axes[2].set_xticklabels(model_names, rotation=35, ha='right', fontsize=8)
axes[2].set_ylabel('Accuracy'); axes[2].legend(); axes[2].grid(axis='y', alpha=0.3)
axes[2].set_title('Exact & Within-1 Accuracy', fontweight='bold')

plt.suptitle('Error, R² & Accuracy Comparison — All Models', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig('10_error_comparison.png'); plt.show()


In [ ]:
# ── 10.4 True vs Predicted scatter plots ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, (name, res) in zip(axes.ravel(), results.items()):
    preds = res['preds']
    rng   = np.random.default_rng(SEED)
    jitter= rng.normal(0, 0.08, len(y_te))
    ax.scatter(y_te + jitter, preds, alpha=0.65, s=40, c=y_te,
               cmap='RdYlGn', vmin=1, vmax=8, edgecolors='white', linewidths=0.3)
    ax.plot([1,8],[1,8], 'k--', lw=1.5, label='Perfect')
    rho = res['metrics']['Spearman_rho']
    ax.set_title(f'{name}  ρ={rho:.3f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('True Score', fontsize=8); ax.set_ylabel('Predicted Score', fontsize=8)
    ax.set_xlim(0.5,8.5); ax.set_ylim(0.5,8.5)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.suptitle('True vs Predicted Score — All Models (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig('11_true_vs_pred.png'); plt.show()


In [ ]:
# ── 10.5 Error distribution histograms ───────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, (name, res) in zip(axes.ravel(), results.items()):
    errors = np.clip(np.round(res['preds']), 1, 8).astype(int) - y_te
    ax.hist(errors, bins=range(-7, 9), color='steelblue', edgecolor='white', align='left')
    ax.axvline(0, color='red', lw=2, linestyle='--')
    ax.axvline(errors.mean(), color='orange', lw=1.5, linestyle='-',
               label=f'mean={errors.mean():.2f}')
    mae = res['metrics']['MAE']
    ax.set_title(f'{name}  MAE={mae:.2f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Prediction Error (pred − true)', fontsize=8)
    ax.set_ylabel('Count', fontsize=8); ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.suptitle('Prediction Error Distribution — All Models (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig('12_error_distributions.png'); plt.show()


In [ ]:
# ── 10.6 Radar chart: multi-metric comparison ────────────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as pe

radar_metrics  = ['Spearman ρ','1 − MAE/7','1 − RMSE/7','R²','Acc@0','Acc@1']

def get_radar_vals(res):
    m = res['metrics']
    return [
        max(0, m['Spearman_rho']),
        max(0, 1 - m['MAE'] / 7),
        max(0, 1 - m['RMSE'] / 7),
        max(0, m['R2']),
        m['Acc@0'],
        m['Acc@1'],
    ]

N = len(radar_metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
palette = plt.cm.tab10(np.linspace(0, 1, len(results)))
for (name, res), col in zip(results.items(), palette):
    vals = get_radar_vals(res) + [get_radar_vals(res)[0]]
    ax.plot(angles, vals, 'o-', linewidth=2, label=name, color=col)
    ax.fill(angles, vals, alpha=0.07, color=col)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=7)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax.set_title('Model Comparison — Radar Chart\n(higher = better for all axes)',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout(); savefig('13_radar_chart.png'); plt.show()


## 11 · Conclusions & Best Model

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────────
print('=' * 65)
print('  FINAL RESULTS — SORTED BY SPEARMAN ρ')
print('=' * 65)
print(df_results.to_string(index=False))
print()

best_model_name = df_results.iloc[0]['Model']
best_metrics    = results[best_model_name]['metrics']
best_params     = results[best_model_name]['best_params']

print(f'\n🏆 Best model: {best_model_name}')
print(f'   Spearman ρ  : {best_metrics["Spearman_rho"]:.4f}')
print(f'   MAE         : {best_metrics["MAE"]:.3f}')
print(f'   RMSE        : {best_metrics["RMSE"]:.3f}')
print(f'   Acc@1       : {best_metrics["Acc@1"]:.1%}')
print(f'   Acc@2       : {best_metrics["Acc@2"]:.1%}')
print(f'   Best params : {best_params}')
print()

# Ordinal cost analysis — how often is error > 2?
print('  Ordinal cost analysis (error > 2 steps = serious mistake):')
for name, res in results.items():
    errors = np.abs(np.clip(np.round(res['preds']), 1, 8).astype(int) - y_te)
    pct_bad = (errors > 2).mean()
    print(f'  {name:25s}: {pct_bad:.1%} predictions off by > 2 steps')


---

## 📝 Summary

| Finding | Details |
|---|---|
| **Dataset** | 213 images, 8 ordinal classes, farmer-labeled (noisy), skewed distribution |
| **Split** | 80/20 train/test — no separate val set; 5-fold CV handles validation |
| **CV design** | Augmentation-aware: folds split by original image, val folds use originals only |
| **Best features** | a\* mean, L\* mean, b\* mean and their distribution shapes (histograms) |
| **Augmentation** | ×16 training set via colour-safe transforms — critical for generalisation |
| **Leakage** | None — split before augmentation, feature pipeline fit per fold only, val folds never see augmented data |
| **Ordinal cost** | Acc@1 and Acc@2 are the primary practical metrics (1-step error acceptable) |

### Next steps
- Consider semi-ordinal ensemble: average Ordinal LR + best regressor predictions
- Investigate misclassified samples — are they the known boundary scores (3/4, 5/6)?
- With more data, revisit the CNN approach (EfficientNetB0 pretrained) from `CNN_ColorCorrection_V4`


## 12. Per-Class Summary Statistics

In [ ]:
# Per-class summary statistics for all models (test set)
print("=" * 80)
print("PER-CLASS SUMMARY STATISTICS (Test Set Predictions)")
print("=" * 80)

for model_name, res in results.items():
    preds = res['preds']

    results_df_model = pd.DataFrame({
        'true_value': y_te.astype(float),
        'predicted_value': preds,
        'absolute_error': np.abs(preds - y_te),
        'squared_error': (preds - y_te) ** 2
    })

    m = res['metrics']
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")

    for target_cls in sorted(results_df_model['true_value'].unique()):
        subset = results_df_model[results_df_model['true_value'] == target_cls]
        mean_pred = subset['predicted_value'].mean()
        std_pred = subset['predicted_value'].std()
        mean_ae = subset['absolute_error'].mean()
        pred_r  = np.clip(np.round(subset['predicted_value'].values), 1, 8).astype(int)
        acc_exact = float((pred_r == int(target_cls)).mean())
        print(f"  Class {int(target_cls):2d}: n={len(subset):3d} | "
              f"mean_pred={mean_pred:.3f} ± {std_pred:.3f} | "
              f"MAE={mean_ae:.3f} | Acc@0={acc_exact:.0%}")

    print(f"\n  OVERALL: R²={m['R2']:.4f}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  "
          f"ρ={m['Spearman_rho']:.4f}  Acc@0={m['Acc@0']:.2%}  Acc@1={m['Acc@1']:.2%}")

    results_df_model.to_csv(
        os.path.join(OUT_DIR, f'{model_name.replace(" ", "_").lower()}_class_results.csv'),
        index=False)

print("\n" + "=" * 80)
print("Per-class results saved to", OUT_DIR)


In [ ]:
# Collect test images for frieze visualization
# X_rgb_filtered[idx_te] gives the filtered test images (uint8 RGB)
X_te_images = X_rgb_filtered[idx_te]
print(f"Test images for frieze: {X_te_images.shape} (uint8 RGB)")

## 13. Draw frieze of predictions

In [ ]:
# Frieze of images arranged by test predictions — all models
# Clean class-grouped layout with colored borders and error indicators

for model_name, res in results.items():
    preds = res['preds']
    order = np.argsort(preds)

    # Subsample every 3rd for clarity
    order = order[::3]
    sub_preds  = preds[order]
    sub_images = X_te_images[order]
    sub_true   = y_te[order]

    # Group by predicted class
    pred_cls = np.clip(np.round(sub_preds), 1, 8).astype(int)
    classes_present = sorted(np.unique(pred_cls))

    # Count separators needed
    n_sep = len(classes_present)
    n_total = len(order) + n_sep
    fig, axes = plt.subplots(1, n_total, figsize=(n_total * 1.8, 5.5))

    pi = 0
    for cls in classes_present:
        # Class separator label
        ax = axes[pi]; ax.axis('off')
        ax.text(0.5, 0.5, f'S{cls}', ha='center', va='center',
                fontsize=22, fontweight='bold',
                color=plt.cm.RdYlGn((cls - 1) / 7.0))
        pi += 1

        # Images in this predicted class
        mask = pred_cls == cls
        for img, pred_v, true_v in zip(sub_images[mask], sub_preds[mask], sub_true[mask]):
            if pi >= n_total: break
            ax = axes[pi]
            ax.imshow(img)
            err = abs(pred_v - true_v)
            col = 'limegreen' if err <= 0.5 else ('orange' if err <= 1.0 else 'red')
            ax.set_title(f'T:{int(true_v)} P:{pred_v:.1f}', fontsize=8,
                         color=col, fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_edgecolor(col); sp.set_linewidth(2)
            pi += 1

    # Hide unused axes
    while pi < n_total:
        axes[pi].axis('off'); pi += 1

    plt.suptitle(f'Prediction Frieze — {model_name}\n'
                 f'🟢 err≤0.5  🟠 err≤1  🔴 err>1',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    savefig(f'frieze_pred_{model_name.replace(" ", "_").lower()}.png')
    plt.show()


In [ ]:
# Frieze of images arranged by TRUE annotations (ground truth ordering)
all_true = y_te.astype(float)

order = np.argsort(all_true)
order = order[::3]  # subsample
sub_true   = all_true[order]
sub_images = X_te_images[order]

# Find best model for predictions overlay
best_model_name = max(results, key=lambda n: results[n]['metrics']['Spearman_rho'])
best_preds_all  = results[best_model_name]['preds']
sub_preds = best_preds_all[np.argsort(all_true)[::3]]

# Group by true class
true_cls = sub_true.astype(int)
classes_present = sorted(np.unique(true_cls))
n_sep = len(classes_present)
n_total = len(order) + n_sep
fig, axes = plt.subplots(1, n_total, figsize=(n_total * 1.8, 5.5))

pi = 0
for cls in classes_present:
    ax = axes[pi]; ax.axis('off')
    ax.text(0.5, 0.5, f'S{cls}', ha='center', va='center',
            fontsize=22, fontweight='bold',
            color=plt.cm.RdYlGn((cls - 1) / 7.0))
    pi += 1

    mask = true_cls == cls
    for img, true_v, pred_v in zip(sub_images[mask], sub_true[mask], sub_preds[mask]):
        if pi >= n_total: break
        ax = axes[pi]
        ax.imshow(img)
        err = abs(pred_v - true_v)
        col = 'limegreen' if err <= 0.5 else ('orange' if err <= 1.0 else 'red')
        ax.set_title(f'T:{int(true_v)} P:{pred_v:.1f}', fontsize=8,
                     color=col, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor(col); sp.set_linewidth(2)
        pi += 1

while pi < n_total:
    axes[pi].axis('off'); pi += 1

plt.suptitle(f'Annotation Frieze (True Labels) — best model overlay: {best_model_name}\n'
             f'🟢 err≤0.5  🟠 err≤1  🔴 err>1',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
savefig('frieze_annotations.png')
plt.show()


## 14. Draw frieze of annotations

In [ ]:
# ── Final Summary & Export ─────────────────────────────────────────────────────
# Save metrics comparison
df_results.to_csv(os.path.join(OUT_DIR, 'model_comparison.csv'), index=False)
print(f'Model comparison saved → {os.path.join(OUT_DIR, "model_comparison.csv")}')

# Print best model
best_name = df_results.iloc[0]['Model']
print(f'\n🏆 Best model: {best_name}')
print(f'   Spearman ρ = {df_results.iloc[0]["Spearman ρ"]:.4f}')
print(f'   R²         = {df_results.iloc[0]["R²"]:.3f}')
print(f'   Acc@0      = {df_results.iloc[0]["Acc@0"]}')
print(f'   Acc@1      = {df_results.iloc[0]["Acc@1"]}')
print(f'   MAE        = {df_results.iloc[0]["MAE"]:.3f}')

# Print overfitting diagnostic
print('\n📋 Overfitting Diagnostic (CV score vs test performance):')
for name, res in results.items():
    m = res['metrics']
    print(f'  {name:20s}: R²={m["R2"]:.3f}  ρ={m["Spearman_rho"]:.3f}  '
          f'Acc@0={m["Acc@0"]:.1%}  MAE={m["MAE"]:.3f}')
